# CT LAB: Simulating and Reconstructiong Tomography Data

Authors: L. Calatroni, A. Sebastiani (MaLGa, Unige)

Mini-course "Computational Imaging & Learning" - MSc in Data Science, University of Padua, Italy.

Welcome to the CT computational lab. The core of the **Computed Tomography (CT)** problem lies in recovering a hidden image from a set of projections. While a single X-ray provides only a flattened shadow of the object of interest, CT measures the total X-ray attenuation along a certain number of different paths.

The goal is to reconstruct precise internal cross-sections, the raw data—represented as a sinogram—consists of discrete, noisy line integrals corrupted by noise.  The objective is to formulate a rigorous forward model of the scanning process based on the discrete Radon transform and solve the resulting ill-posed inverse problem to recover the tissue attenuation map. We will utilize the `deepinv` library to implement optimization algorithms for Maximum Likelihood and Maximum a Posteriori estimation.

*First, run the setup cell below to install the necessary libraries and load all the tools.*

In [ ]:
!pip install deepinv

In [ ]:
import numpy as np

# Plots
import matplotlib.pyplot as plt

# Importing images and basic operations
from skimage.data import shepp_logan_phantom
from skimage.color import rgb2gray
from skimage.transform import resize

import deepinv as dinv
import torch

# Check if GPU is available
device = dinv.utils.get_freer_gpu() if torch.cuda.is_available() else "cpu"
print(f'Device is {device}')

# Use parallel dataloader if using a GPU to fasten training.
num_workers = 5 if torch.cuda.is_available() else 0

dtype = torch.float32
circle = False

MSE = dinv.loss.metric.MSE()
PSNR = dinv.loss.metric.PSNR()
SSIM = dinv.loss.metric.SSIM()

# Image loading

In [ ]:
# A benchmark image for medical imaging problems
u_SL = shepp_logan_phantom()

# Reshape the image to a smaller size (with piecewise constant interpolation)
N_SL_small = 128
u_SL_small = resize(u_SL, (N_SL_small,N_SL_small), order=0,preserve_range=True,anti_aliasing=True)


u_SL = torch.tensor(u_SL).unsqueeze(0).unsqueeze(0).to(dtype).to(device)
u_SL_small = torch.tensor(u_SL_small).unsqueeze(0).unsqueeze(0).to(dtype).to(device)


dinv.utils.plot([u_SL,u_SL_small], ['Full image', 'Small image'], figsize=(6,4))

# **Radon Transform - Computed Tomography**

Collect integrals of an image $f$ along straight lines:

$$ \begin{aligned} \mathbf{y}(\boldsymbol{\theta},\rho) &= \int_{L_{\boldsymbol{\theta},\rho}}~u~d\sigma
\end{aligned} \qquad \boldsymbol{\theta} \in \R^2,~ \| \boldsymbol{\theta}\|=1, \quad \rho \in \mathbb{R}_+$$

($\mathbf{y}$ is usually referred to as the *sinogram* of $u$ and $L_{\boldsymbol{\theta},\rho} =  \left\{ \mathbf{x}: \mathbf{x}\cdot \boldsymbol{\theta} = \rho \right\}$)

Subsampling can be motivated by physical limitations in the acquisition procedure and by the goal of reducing the exposition to X-rays
*   **sparse** angles: (uniformly) subsample the angle $\omega$ such that $\boldsymbol{\theta} = (\cos\omega, \sin\omega)$.
*   **limited** angles: acquire angles only in a limited wedge $[\omega_{\text{min}},\omega_{\text{max}}]$

# Defining the Forward model

In [ ]:
# define a forward model with 180 angles in the full angular range [0, 180]
img_width = N_SL_small
N_theta_full = # TODO
theta_full = # TODO np.linspace(#, #, #, endpoint=False)
theta_full = torch.tensor(theta_full)

tomo_full = dinv.physics.Tomography(angles=theta_full, img_width=img_width, circle=circle, device=device)

sinogram_full = tomo_full(u_SL_small)

In [ ]:
# define a sparse angle CT forward model with 36 angles in the full angular range [0, 180]
N_theta_sparse = # TODO
theta_sparse = # TODO np.linspace(#, #, #, endpoint=False)
theta_sparse = torch.tensor(theta_sparse)

tomo_sparse = dinv.physics.Tomography(angles=theta_sparse, img_width=img_width, circle=circle, device=device)

sinogram_sparse = tomo_sparse(u_SL_small)

In [ ]:
# define a limited angle CT forward model within the angular range [30, 150]
Theta_min = # TODO
Theta_max = # TODO
N_theta_lim = np.round((Theta_max-Theta_min)/180*N_theta_full).astype('int')
theta_limited = # TODO
theta_limited = torch.tensor(theta_limited)

tomo_limited = dinv.physics.Tomography(angles=theta_limited, img_width=img_width, circle=circle, device=device)

sinogram_limited = tomo_limited(u_SL_small)

In [ ]:
# fill the entire sinograms with the limited data sinograms
fill_sinogram_sparse = torch.zeros_like(sinogram_full)
fill_sinogram_sparse[..., theta_sparse.int()]=# TODO

fill_sinogram_limited = torch.zeros_like(sinogram_full)
fill_sinogram_limited[..., theta_limited.int()]=# TODO

img_list = [u_SL_small, sinogram_full, fill_sinogram_sparse, fill_sinogram_limited]
title_list = ['Image', 'Full sinogram', 'Sparse-angle sinogram', 'Limited-angle sinogram']

dinv.utils.plot(img_list, title_list, figsize=(12,4))

In [6]:
# For adding Gaussian noise
class GaussianNoiseCT(dinv.physics.GaussianNoise):
  def __init__(
        self,
        sigma: float | torch.Tensor = 0.1,
        rng: torch.Generator | None = None,
    ):
        super().__init__(sigma=sigma, rng=rng)

  def forward(self, x, sigma=None, seed=None, **kwargs):
        self.update_parameters(sigma=sigma, **kwargs)
        self.to(x.device)
        return (
            x
            + torch.max(torch.abs(x))*self.randn_like(x, seed=seed)
            * self.sigma[(...,) + (None,) * (x.dim() - 1)]
        )

# **Näive inversion**

An inverse problem can be representing as recovering $f$ from
$$ y = \operatorname{noise}(A(u))$$
Let $A$ be linear (and additive noise): in the discrete formulation, this reduces to
$$ y = \mathbf{A} u + ϵ, \quad u \in \mathbb{R}^n, \epsilon \in \mathbb{R}^m, \mathbf{A} \in \mathbb{R}^{m \times n} $$

Easy idea to solve it:
$$ u_{\text{naive}} = \mathbf{A}^{-1} y$$

For the Radon tranform, we used the Filtered Back-Projection (FBP) to ``invert'' the measurements.

**TASK 1: Experiment with the Forward Model**

Change the variables in the cells above and re-run them to see the effects!
* **Change the phisyics:** Modify `tomo_exp` (physics) to see how the size of the sinogram changes.
* **Change the noise:** Adjust the `nl` parameter. How does the image look when the noise level increase?
* **Observe the reconstructions:** Compare the reconstructions obtained with the FBP under different acquisition geometries.

In [ ]:
# Selection operator for experiments, select any of the previous ones
tomo_exp = # TODO

In [ ]:
sinogram = tomo_exp(u_SL_small)
u_rec = tomo_exp.fbp(sinogram)

# add noise to the sinogram with noise level 1e-3
nl1 = # TODO
noise_ph = # TODO
sinogram_noisy = noise_ph(sinogram)
# compute the FBP of the noisy sinogram
u_rec_noisy = # TODO

# add noise to the sinogram with noise level 1e-2
nl2 = # TODO
noise_ph2 = # TODO
sinogram_noisy2 = noise_ph2(sinogram)
# compute the FBP of the noisy sinogram
u_rec_noisy2 = # TODO

titles_sino = ['Noiseless', f'nl = {nl1}', f'nl = {nl2}']
titles_rec = ['Noiseless recon', f'Noisy recon nl = {nl1}', f'Noisy recon nl = {nl2}']

dinv.utils.plot([sinogram, sinogram_noisy, sinogram_noisy2], titles_sino, figsize=(12,4))
dinv.utils.plot([u_rec, u_rec_noisy, u_rec_noisy2], titles_rec, figsize=(12,4))

In [ ]:
# Compute the quality metrics for the reconstructed images
print(f'Reconstruction (noiseless sinogram)')
print(f' MSE={MSE(u_rec, u_SL_small).item():.4f}\n PSNR={PSNR(u_rec, u_SL_small).item():.2f}\n SSIM={SSIM(u_rec, u_SL_small).item():.2f}\n')

print(f'Reconstruction (sinogram noise lev {nl1})')
print(f' MSE={MSE(u_rec_noisy, u_SL_small).item():.4f}\n PSNR={PSNR(u_rec_noisy, u_SL_small).item():.2f}\n SSIM={SSIM(u_rec_noisy, u_SL_small).item():.2f}\n')

print(f'Reconstruction (sinogram noise lev {nl2})')
print(f' MSE={MSE(u_rec_noisy2, u_SL_small).item():.4f}\n PSNR={PSNR(u_rec_noisy2, u_SL_small).item():.2f}\n SSIM={SSIM(u_rec_noisy2, u_SL_small).item():.2f}\n')

In [ ]:
# Selection corrupted sinogram for the following experiments
y = sinogram_noisy2

# Unregularized Reconstruction
Ideally, we would like to solve the system $\mathbf{A} u = y$. Assembling the matrix $\mathbf{A}$ is very expensive in terms of memory.
We can avoid assembling the matrix $\mathbf{A}$!

Iterative solvers for $\mathbf{A} u = y$: e.g. minimizing $\frac{1}{2}\| \mathbf{A} u- y\|^2$ via **gradient method**:

$$
\left\{
\begin{aligned}
u^{0} & \text{ given} \\
u^{(k+1)} =&\  \mathbf{A}^T(\mathbf{A}u^{(k)}-y)
\end{aligned}
\right.
$$

This only requires the knowledge of $\mathbf{A}$ and its adjoint $\mathbf{A}^T$. The application of $\mathbf{A}$ and $\mathbf{A}^T$ can be done without matrices via operators, functions...

In [ ]:
# Define Fidelity
fidelity = dinv.optim.data_fidelity.L2()

# Define Prior
#prior = dinv.optim.prior.Zero()
prior = dinv.optim.prior.ZeroPrior()

# Define optimizer (check https://deepinv.github.io/deepinv/api/stubs/deepinv.optim.optim_builder.html)
opt = dinv.optim.optim_builder() #TODO
# Run
u_hat, metrics = opt(y, tomo_exp, compute_metrics=True)

print(f'Reconstruction experiment')
print(f' MSE={MSE(u_hat, u_SL_small).item():.4f}\n PSNR={PSNR(u_hat, u_SL_small).item():.2f}\n SSIM={SSIM(u_hat, u_SL_small).item():.2f}\n')

plt.plot(metrics['cost'][0])
dinv.utils.plot([u_hat, u_SL_small], ['Recons itr', 'GT'], figsize=(12,4))

# Tikhonov regularization
Penalizes solutions with large norms:

$$ u_{\alpha}^* = \arg\min_{u \in \mathbf{R}^n} \left\{ \frac{1}{2}\| A u - y \|^2 + \alpha \| u\|^2\right\}$$

(the first term of the sum measures the data fidelity and might be chosen differently according to the noise model).

The solution can be also found via first-order optimality conditions:

$$  A^T(A u_{\alpha} -y) + \alpha u_{\alpha}^* = 0 \quad ⇒ \quad (A^T A + \alpha \operatorname{Id})u_{\alpha}^* = A^T y $$

thus $u_\alpha^*$ can be found solving a linear system with symmetric, positive definite, matrix.

In [ ]:
# Define Fidelity
fidelity = dinv.optim.data_fidelity.L2()

# Define Prior
prior = dinv.optim.Tikhonov()

# Define optimizer
opt = dinv.optim.optim_builder() #TODO
## Run
u_hat, metrics = opt(y, tomo_exp, compute_metrics=True)

print(f'Reconstruction experiment')
print(f' MSE={MSE(u_hat, u_SL_small).item():.4f}\n PSNR={PSNR(u_hat, u_SL_small).item():.2f}\n SSIM={SSIM(u_hat, u_SL_small).item():.2f}\n')

plt.plot(metrics['cost'][0])
dinv.utils.plot([u_hat, u_SL_small], ['Recons itr', 'GT'], figsize=(12,4))

# Sparsity-promoting regularization (LASSO)
Penalizes solutions with large $1$-norms: this can be seen as the convex relaxation on the $0$-'norm' penalization, hence promoting the fact that the solution has **few pixels different from 0**

$$ u_{\alpha}^* = \arg\min_{u \in \mathbf{R}^n} \left\{ \frac{1}{2}\| Au - y \|^2 + \alpha \| u\|_1 \right\}$$

Approximate $u_\alpha$ via **Proximal-Gradient Descent** (**PGD**) method: the proximal of $\alpha \| \cdot \|_1$ is **soft-thresholding** $S_\alpha(u) = \operatorname{sign}(u) (u-\alpha)^+$. This version of PGD is known as ISTA (Iterative Soft-Thresholding Algorithm)

$$
\left\{
\begin{aligned}
u^{0}   & \quad \text{given} \\
u^{k+1} &= S_{\tau \alpha} \big(u^{k} - \tau A^T(A u^{k}-y)\big)
\end{aligned}
\right.
$$

In [ ]:
# Define Fidelity
fidelity = dinv.optim.data_fidelity.L2()

# Define Prior
prior = dinv.optim.L1Prior()

# Define optimizer
opt = dinv.optim.optim_builder() #TODO
## Run
u_hat, metrics = opt(y, tomo_exp, compute_metrics=True)

print(f'Reconstruction experiment')
print(f' MSE={MSE(u_hat, u_SL_small).item():.4f}\n PSNR={PSNR(u_hat, u_SL_small).item():.2f}\n SSIM={SSIM(u_hat, u_SL_small).item():.2f}\n')

plt.plot(metrics['cost'][0])
dinv.utils.plot([u_hat, u_SL_small], ['Recons itr', 'GT'], figsize=(12,4))

**TASK 2: Hyperparameter search**

Run the algorithms above. Play with the `stepsize`, `lambda` and `max_iter` hyperparameters to see how they affect the reconstruction. Does running it for more iterations always make the image better?

_Use code below to make see the differences in terms of MSE, PSNR and SSIM changing the parameters_

In [ ]:
MSE_list = []
PSNR_list = []
SSIM_list = []
par_list = np.linspace(1e-3, 1, num=5)

for par in par_list:
    opt = dinv.optim.optim_builder(iteration="GD", prior=prior, data_fidelity=fidelity, \
                                    max_iter=100, crit_conv='residual', thres_conv=1e-4, early_stop=True, \
                                    params_algo={"stepsize": par}
                                  )
    u_hat = opt(y, tomo_exp)

    MSE_list.append(MSE(u_hat, u_SL_small))
    PSNR_list.append(PSNR(u_hat, u_SL_small))
    SSIM_list.append(SSIM(u_hat, u_SL_small))

fig, axs = plt.subplots(1, 3)
axs[0].plot(par_list, MSE_list)
axs[1].plot(par_list, PSNR_list)
axs[2].plot(par_list, SSIM_list)
plt.show()

**Optional TASK: Construction A**
- Assembly the matrix $\mathbf{A}$ associated to the Radon transform (use a small `image_size` and a sparse geometry at first).
- Compute the SVD observing the decay of its singular values.

In [ ]:
N = 30
N_angles = 30
theta = np.linspace(0, 180, N_angles, endpoint=False)
theta = torch.tensor(theta)
tomoSVD = dinv.physics.Tomography(angles=theta, img_width=N, circle=circle, device=device)

temp = tomoSVD(torch.zeros((1,1,N,N)))
M = torch.numel(temp)

In [ ]:
A_mat = #TODO

for i in range(...):
  e_i = #TODO
  e_i[i] = 1
  e_i = e_i.view(1,1,N,N)
  A_mat[:, i] =  torch.reshape(...,(M,))

In [ ]:
dinv.utils.plot(A_mat.unsqueeze(0), ['A_mat'], figsize=(6,4))

In [ ]:
S = torch.linalg.svdvals(A_mat)

In [ ]:
plt.semilogy(S.numpy())
plt.show()

In [ ]:
# TASK: observe the decay of the singular values in case of Radon, limited-angle Radon, sparse-angle Radon